In [ ]:
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import load_model

# ─── Hardcoded Paths ────────────────────────────────────────────────────────
MODEL_PATH   = '/Users/ashishkishore/Downloads/final_symptom_model.keras'
SCALER_PATH  = '/Users/ashishkishore/Downloads/symptom_scaler.pkl'
ENCODER_PATH = '/Users/ashishkishore/Downloads/symptom_label_encoder.pkl'
DATASET_PATH = '/Users/ashishkishore/Downloads/symbipredict_2022.csv'

# ─── Load Artifacts ─────────────────────────────────────────────────────────
model = load_model(MODEL_PATH)

with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)

with open(ENCODER_PATH, 'rb') as f:
    le = pickle.load(f)

dataset  = pd.read_csv(DATASET_PATH)
SYMPTOMS = dataset.drop('prognosis', axis=1).columns.tolist()

print(f"Model loaded | {len(SYMPTOMS)} symptoms | {len(le.classes_)} diseases")

# ─── Predict ────────────────────────────────────────────────────────────────
def predict(symptom_names: list, top_k: int = 5):
    vec = np.zeros(len(SYMPTOMS))
    not_found = []
    for s in symptom_names:
        s = s.strip().lower()
        if s in SYMPTOMS:
            vec[SYMPTOMS.index(s)] = 1.0
        else:
            not_found.append(s)

    if not_found:
        print(f"\nNot recognised (check spelling): {not_found}")
        close = [sym for nf in not_found for sym in SYMPTOMS if nf[:5] in sym]
        if close:
            print(f"Did you mean? -> {list(set(close))[:6]}")

    X     = np.expand_dims(scaler.transform([vec]), -1)
    probs = model.predict(X, verbose=0)[0]
    top_idx = np.argsort(probs)[::-1][:top_k]

    print(f"\n{'─' * 50}")
    for rank, idx in enumerate(top_idx, 1):
        marker = "=>" if rank == 1 else "  "
        print(f" {marker} {rank}. {le.classes_[idx]:<35} {probs[idx]*100:5.1f}%")
    print(f"{'─' * 50}")

    best = top_idx[0]
    return le.classes_[best], round(float(probs[best]) * 100, 2)

# ─── Main Loop ──────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print("\n" + "=" * 50)
    print("       DISEASE PREDICTION FROM SYMPTOMS")
    print("=" * 50)
    print("  - Enter symptoms separated by commas")
    print("  - Type 'list' to see all valid symptoms")
    print("  - Type 'search <keyword>' to find a symptom")
    print("  - Type 'quit' to exit")
    print("=" * 50)

    while True:
        raw = input("\nEnter symptoms: ").strip()

        if raw.lower() == 'quit':
            print("Goodbye!")
            break

        elif raw.lower() == 'list':
            print(f"\nAll {len(SYMPTOMS)} valid symptoms:")
            for i, s in enumerate(SYMPTOMS, 1):
                print(f"  {i:3}. {s}")

        elif raw.lower().startswith('search '):
            keyword = raw[7:].strip()
            matches = [s for s in SYMPTOMS if keyword.lower() in s]
            print(f"\nMatches for '{keyword}': {matches if matches else 'None found'}")

        elif raw == '':
            print("Please enter at least one symptom.")

        else:
            symptoms = [s.strip() for s in raw.split(',')]
            disease, confidence = predict(symptoms)
            print(f"\n  Final -> {disease} ({confidence}%)")

In [3]:
"""
SYMPTOM MODEL — CLINICAL STRESS TEST SUITE
============================================
12 tests designed the way a doctor validates a diagnostic AI:
 1.  Training distribution baseline
 2.  External dataset baseline (dataset.csv)
 3.  Confusable disease pairs
 4.  Minimum symptom threshold (early-stage patient)
 5.  Atypical presentation (rare symptoms only)
 6.  Related symptom injection (plausible extras for same disease)
 7.  Cross-contamination (inject symptoms from most similar disease)
 8.  Early / Mid / Late stage simulation
 9.  Dangerous misclassification audit (critical diseases)
 10. Symptom dropout (per-symptom importance)
 11. Progressive unrelated noise
 12. Confidence calibration + speed
"""

import numpy as np
import pandas as pd
import pickle, time, os
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from sklearn.metrics import (classification_report, confusion_matrix,
                             top_k_accuracy_score, f1_score)
from itertools import combinations
from collections import Counter

# ─── Paths ───────────────────────────────────────────────────────────────────
MODEL_PATH    = '/Users/ashishkishore/Downloads/final_symptom_model.keras'
SCALER_PATH   = '/Users/ashishkishore/Downloads/symptom_scaler.pkl'
ENCODER_PATH  = '/Users/ashishkishore/Downloads/symptom_label_encoder.pkl'
TRAIN_DATASET = '/Users/ashishkishore/Downloads/symbipredict_2022.csv'
TEST_DATASET  = '/Users/ashishkishore/Downloads/dataset.csv'
RESULTS_DIR   = '/Users/ashishkishore/Downloads/stress_test_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─── Load ────────────────────────────────────────────────────────────────────
print("Loading model...")
model  = load_model(MODEL_PATH)
scaler = pickle.load(open(SCALER_PATH,  'rb'))
le     = pickle.load(open(ENCODER_PATH, 'rb'))

train_df = pd.read_csv(TRAIN_DATASET)
SYMPTOMS  = train_df.drop('prognosis', axis=1).columns.tolist()   # 132 symptoms
NC        = len(le.classes_)
SYM_SET   = set(SYMPTOMS)
print(f"Model loaded | {len(SYMPTOMS)} symptoms | {NC} diseases")
print(f"Known classes: {list(le.classes_[:5])} ...")

# ─── Auto-build disease mapping from dataset.csv ─────────────────────────────
# dataset.csv disease names are used directly — identity map where possible,
# fuzzy-strip for any whitespace differences.
test_df_raw = pd.read_csv(TEST_DATASET)
le_classes_set = set(le.classes_)

def auto_map(disease_str):
    """Return the le.classes_ name for a dataset.csv disease string, or None."""
    if disease_str in le_classes_set:
        return disease_str
    stripped = disease_str.strip()
    if stripped in le_classes_set:
        return stripped
    # try adding/removing trailing space
    for candidate in [disease_str + ' ', disease_str.rstrip() ]:
        if candidate in le_classes_set:
            return candidate
    return None  # truly unknown

DISEASE_MAP = {}
unmapped = []
for d in test_df_raw['Disease'].unique():
    mapped = auto_map(d)
    if mapped:
        DISEASE_MAP[d] = mapped
    else:
        unmapped.append(d)

print(f"\nDisease mapping: {len(DISEASE_MAP)} matched, {len(unmapped)} excluded")
if unmapped:
    print(f"  Excluded (not in label encoder): {unmapped}")

# ─── Build disease knowledge base from TRAINING data ─────────────────────────
# For each disease: symptom frequency across all training rows
DSYM_FREQ  = {}   # {disease: {symptom: freq}}  — all symptoms with freq>0
DSYM_POOL  = {}   # freq >= 0.10  (common enough to be plausible)
DSYM_CORE  = {}   # freq >= 0.80  (almost always present — textbook symptoms)
DSYM_RARE  = {}   # 0 < freq < 0.40  (uncommon — atypical)

for disease, group in train_df.groupby('prognosis'):
    X    = group.drop('prognosis', axis=1).values.astype(float)
    freq = X.mean(axis=0)
    DSYM_FREQ[disease] = {SYMPTOMS[i]: freq[i] for i in range(len(SYMPTOMS)) if freq[i] > 0}
    DSYM_POOL[disease] = [SYMPTOMS[i] for i in range(len(SYMPTOMS)) if freq[i] >= 0.10]
    DSYM_CORE[disease] = [SYMPTOMS[i] for i in range(len(SYMPTOMS)) if freq[i] >= 0.80]
    DSYM_RARE[disease] = [SYMPTOMS[i] for i in range(len(SYMPTOMS)) if 0 < freq[i] < 0.40]

# ─── Build confusable pairs (top 20 by Jaccard on DSYM_POOL) ─────────────────
def build_confusable_pairs(top_n=20):
    pairs = []
    diseases = list(DSYM_POOL.keys())
    for d1, d2 in combinations(diseases, 2):
        s1, s2 = set(DSYM_POOL[d1]), set(DSYM_POOL[d2])
        if not s1 | s2: continue
        j      = len(s1 & s2) / len(s1 | s2)
        shared = sorted(s1 & s2)
        only1  = sorted(s1 - s2)   # symptoms unique to d1 (discriminating)
        only2  = sorted(s2 - s1)   # symptoms unique to d2
        pairs.append((d1, d2, j, shared, only1, only2))
    pairs.sort(key=lambda x: -x[2])
    return pairs[:top_n]

CONFUSABLE = build_confusable_pairs()

# ─── Core helpers ─────────────────────────────────────────────────────────────
def wide_to_binary(df):
    """Convert Symptom_1..17 wide format → binary (n, 132) matrix."""
    sym_cols = [c for c in df.columns if c.startswith('Symptom_')]
    X = np.zeros((len(df), len(SYMPTOMS)), dtype=float)
    for idx, row in df.iterrows():
        for col in sym_cols:
            val = row[col]
            if isinstance(val, str):
                sym = val.strip().lower()
                if sym in SYM_SET:
                    X[idx, SYMPTOMS.index(sym)] = 1.0
    y_raw      = df['Disease'].map(DISEASE_MAP)
    known      = y_raw.notna()
    return X[known.values], y_raw[known].values, known

def infer(X_raw):
    return model.predict(np.expand_dims(scaler.transform(X_raw), -1), verbose=0)

def score(y, probs, tag=''):
    preds = np.argmax(probs, 1)
    acc   = (preds == y).mean()
    top3  = top_k_accuracy_score(y, probs, k=3, labels=range(NC))
    top5  = top_k_accuracy_score(y, probs, k=5, labels=range(NC))
    f1    = f1_score(y, preds, average='macro', zero_division=0)
    conf  = np.max(probs, 1).mean()
    if tag: print(f"\n  [{tag}]")
    print(f"    Acc={acc:.4f}  Top3={top3:.4f}  Top5={top5:.4f}  "
          f"F1={f1:.4f}  AvgConf={conf:.4f}")
    return dict(tag=tag, acc=acc, top3=top3, top5=top5, f1=f1, conf=conf)

def bar_chart(labels, accs, f1s, title, path):
    x = np.arange(len(labels))
    fig, axes = plt.subplots(1, 2, figsize=(max(12, len(labels)), 5))
    axes[0].bar(x, accs, color='#2E75B6', alpha=0.85)
    axes[0].axhline(accs[0], color='red', ls='--', lw=1.5, label='Baseline')
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    axes[0].set_ylim(0, 1.08); axes[0].set_title('Accuracy'); axes[0].legend()
    axes[0].grid(alpha=0.3, axis='y')
    axes[1].bar(x, f1s, color='#27AE60', alpha=0.85)
    axes[1].axhline(f1s[0], color='red', ls='--', lw=1.5, label='Baseline')
    axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    axes[1].set_ylim(0, 1.08); axes[1].set_title('Macro F1'); axes[1].legend()
    axes[1].grid(alpha=0.3, axis='y')
    plt.suptitle(title, fontweight='bold'); plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches='tight'); plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# TEST 1 — TRAINING DISTRIBUTION BASELINE
# ══════════════════════════════════════════════════════════════════════════════
def test1_training_baseline():
    print("\n" + "="*65)
    print("  TEST 1: Training Distribution Baseline (symbipredict_2022)")
    print("="*65)
    df = pd.read_csv(TRAIN_DATASET)
    X  = df.drop('prognosis', axis=1).values.astype(float)
    y  = le.transform(df['prognosis'].values)
    t0 = time.time()
    probs = infer(X)
    print(f"  Samples={len(y)}  Speed={len(y)/(time.time()-t0):.0f} samp/sec")
    m = score(y, probs, 'Baseline — symbipredict')
    cm = confusion_matrix(y, np.argmax(probs,1))
    plt.figure(figsize=(16,13))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title('TEST 1 — Training Distribution CM')
    plt.xticks(rotation=90, fontsize=6); plt.yticks(fontsize=6)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/test1_train_cm.png', dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  Saved: test1_train_cm.png\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 2 — EXTERNAL DATASET BASELINE (dataset.csv, no modification)
# ══════════════════════════════════════════════════════════════════════════════
def test2_external_baseline():
    print("="*65)
    print("  TEST 2: External Dataset Baseline (dataset.csv — unmodified)")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X, y_raw, _ = wide_to_binary(df)
    y = le.transform(y_raw)
    print(f"  Usable samples={len(y)}")
    probs = infer(X)
    m = score(y, probs, 'External baseline')

    present = np.unique(y)
    names   = le.classes_[present]
    report  = classification_report(y, np.argmax(probs,1), labels=present,
                                    target_names=names, output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(f'{RESULTS_DIR}/test2_per_class.csv')

    # Per-disease F1 sorted
    f1s = {r: report[r]['f1-score'] for r in names if r in report}
    sf  = dict(sorted(f1s.items(), key=lambda x: x[1]))
    colors = ['#C0392B' if v<0.7 else '#F39C12' if v<0.9 else '#27AE60' for v in sf.values()]
    plt.figure(figsize=(10,8))
    plt.barh(list(sf.keys()), list(sf.values()), color=colors)
    plt.axvline(0.9, color='green', ls='--', alpha=0.6, label='0.90')
    plt.axvline(0.7, color='red',   ls='--', alpha=0.6, label='0.70')
    plt.xlabel('F1 Score'); plt.title('TEST 2 — Per-Disease F1 (External)')
    plt.legend(); plt.grid(alpha=0.3, axis='x'); plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/test2_per_disease_f1.png', dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  Saved: test2_per_class.csv + test2_per_disease_f1.png\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 3 — CONFUSABLE DISEASE PAIRS
# Which disease pairs share the most symptoms?
# Does the model confuse them when shown shared symptoms only?
# ══════════════════════════════════════════════════════════════════════════════
def test3_confusable_pairs():
    print("="*65)
    print("  TEST 3: Confusable Disease Pairs")
    print("  (Tests if model distinguishes diseases with overlapping symptoms)")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)

    rows = []
    print(f"\n  {'Disease A':<32} {'Disease B':<32} {'Jac':>5} "
          f"{'AccA':>6} {'AccB':>6} {'A->B':>6} {'B->A':>6}")
    print(f"  {'-'*95}")

    for d1, d2, jac, shared, only1, only2 in CONFUSABLE:
        # Map to encoder
        enc1 = auto_map(d1); enc2 = auto_map(d2)
        if not enc1 or not enc2: continue
        if enc1 not in le.classes_ or enc2 not in le.classes_: continue
        idx1 = np.where(le.classes_==enc1)[0][0]
        idx2 = np.where(le.classes_==enc2)[0][0]
        m1   = y_all==idx1; m2 = y_all==idx2
        if m1.sum()==0 or m2.sum()==0: continue

        p1   = np.argmax(infer(X_all[m1]),1)
        p2   = np.argmax(infer(X_all[m2]),1)
        acc1 = (p1==idx1).mean(); acc2 = (p2==idx2).mean()
        a2b  = (p1==idx2).mean(); b2a  = (p2==idx1).mean()

        print(f"  {d1:<32} {d2:<32} {jac:>5.3f} "
              f"{acc1:>6.3f} {acc2:>6.3f} {a2b:>6.3f} {b2a:>6.3f}")
        rows.append({'disease_a':d1,'disease_b':d2,'jaccard':jac,
                     'acc_a':round(acc1,4),'acc_b':round(acc2,4),
                     'a_predicted_as_b':round(a2b,4),'b_predicted_as_a':round(b2a,4),
                     'shared_symptoms':len(shared),
                     'unique_to_a':len(only1),'unique_to_b':len(only2)})

    pd.DataFrame(rows).to_csv(f'{RESULTS_DIR}/test3_confusable.csv', index=False)
    print(f"\n  Saved: test3_confusable.csv\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 4 — MINIMUM SYMPTOM THRESHOLD
# Simulate early-stage patients who present with very few symptoms.
# Keeps the TOP-N most frequent symptoms for each disease row.
# ══════════════════════════════════════════════════════════════════════════════
def test4_min_symptoms():
    print("="*65)
    print("  TEST 4: Minimum Symptom Threshold")
    print("  (Early-stage patient — only N most common symptoms present)")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)
    results = []
    print(f"\n  {'N symptoms kept':<20} {'Acc':>8} {'Top3':>8} {'F1':>8}")
    print(f"  {'-'*46}")

    for n_keep in [1, 2, 3, 4, 5, 7, 10, 'all']:
        X_aug = np.zeros_like(X_all)
        for i in range(len(X_all)):
            on_idx   = np.where(X_all[i]==1)[0]
            if len(on_idx)==0: continue
            disease  = y_raw_all[i]
            freq_map = DSYM_FREQ.get(disease, {})
            # Sort ON symptoms by training frequency (keep most diagnostic ones)
            ranked = sorted(on_idx,
                            key=lambda j: freq_map.get(SYMPTOMS[j], 0),
                            reverse=True)
            keep = ranked if n_keep=='all' else ranked[:min(n_keep, len(ranked))]
            X_aug[i, keep] = 1.0

        probs = infer(X_aug)
        preds = np.argmax(probs,1)
        acc   = (preds==y_all).mean()
        top3  = top_k_accuracy_score(y_all, probs, k=3, labels=range(NC))
        f1    = f1_score(y_all, preds, average='macro', zero_division=0)
        label = f'Top-{n_keep}' if n_keep!='all' else 'All'
        print(f"  {label:<20} {acc:>8.4f} {top3:>8.4f} {f1:>8.4f}")
        results.append({'n_keep':n_keep,'acc':round(acc,4),
                        'top3':round(top3,4),'f1':round(f1,4)})

    rdf = pd.DataFrame(results)
    rdf.to_csv(f'{RESULTS_DIR}/test4_min_symptoms.csv', index=False)
    num = rdf[rdf['n_keep']!='all'].copy()
    num['n_keep'] = num['n_keep'].astype(int)
    plt.figure(figsize=(8,5))
    plt.plot(num['n_keep'],num['acc'],'b-o',ms=7,lw=2,label='Accuracy')
    plt.plot(num['n_keep'],num['top3'],'g-s',ms=7,lw=2,label='Top-3')
    plt.plot(num['n_keep'],num['f1'],'r-^',ms=7,lw=2,label='Macro F1')
    plt.xlabel('N symptoms kept (most frequent first)')
    plt.ylabel('Score'); plt.title('TEST 4 — Minimum Symptom Threshold')
    plt.legend(); plt.grid(alpha=0.3); plt.ylim(0,1.05); plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/test4_min_symptoms.png', dpi=200)
    plt.close()
    print(f"\n  Saved: test4_min_symptoms.csv + .png\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 5 — ATYPICAL PRESENTATION
# Patient presents with ONLY rare/uncommon symptoms (not the textbook ones).
# Real patients often don't present classically — this tests robustness.
# ══════════════════════════════════════════════════════════════════════════════
def test5_atypical():
    print("="*65)
    print("  TEST 5: Atypical Presentation")
    print("  (Patient shows only RARE symptoms — not textbook/common ones)")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)

    X_core   = np.zeros_like(X_all)  # only >=80% freq symptoms
    X_rare   = np.zeros_like(X_all)  # only <40% freq symptoms
    X_mid    = np.zeros_like(X_all)  # 40-80% freq

    for i in range(len(X_all)):
        disease  = y_raw_all[i]
        on_idx   = np.where(X_all[i]==1)[0]
        freq_map = DSYM_FREQ.get(disease, {})
        core = [j for j in on_idx if freq_map.get(SYMPTOMS[j],0)>=0.80]
        rare = [j for j in on_idx if freq_map.get(SYMPTOMS[j],0)<0.40]
        mid  = [j for j in on_idx if 0.40<=freq_map.get(SYMPTOMS[j],0)<0.80]
        # Fallback if no symptoms in that frequency band
        if core: X_core[i, core] = 1.0
        else:    X_core[i, on_idx[:min(2,len(on_idx))]] = 1.0
        if rare: X_rare[i, rare] = 1.0
        else:    X_rare[i, on_idx[-min(2,len(on_idx)):]] = 1.0
        if mid:  X_mid[i, mid]  = 1.0
        else:    X_mid[i, on_idx[:min(3,len(on_idx))]] = 1.0

    m_all  = score(y_all, infer(X_all),  'Full symptom set')
    m_core = score(y_all, infer(X_core), 'Core/textbook symptoms only (freq>=80%)')
    m_mid  = score(y_all, infer(X_mid),  'Mid-frequency symptoms (40-80%)')
    m_rare = score(y_all, infer(X_rare), 'Atypical/rare symptoms only (freq<40%)')

    # Per-disease accuracy under atypical presentation
    rows = []
    for disease_raw in sorted(np.unique(y_raw_all)):
        enc = auto_map(disease_raw)
        if not enc or enc not in le.classes_: continue
        idx  = np.where(le.classes_==enc)[0][0]
        mask = y_all==idx
        if mask.sum()==0: continue
        a_full = (np.argmax(infer(X_all[mask]),1)==idx).mean()
        a_core = (np.argmax(infer(X_core[mask]),1)==idx).mean()
        a_rare = (np.argmax(infer(X_rare[mask]),1)==idx).mean()
        rows.append({'disease':disease_raw,'acc_full':round(a_full,4),
                     'acc_core':round(a_core,4),'acc_atypical':round(a_rare,4),
                     'drop_atypical':round(a_full-a_rare,4)})

    rdf = pd.DataFrame(rows).sort_values('drop_atypical', ascending=False)
    rdf.to_csv(f'{RESULTS_DIR}/test5_atypical.csv', index=False)
    print(f"\n  Top 10 most fragile to atypical presentation:")
    print(f"  {'Disease':<35} {'Full':>6} {'Core':>6} {'Rare':>6} {'Drop':>6}")
    print(f"  {'-'*57}")
    for _, r in rdf.head(10).iterrows():
        print(f"  {r['disease']:<35} {r['acc_full']:>6.3f} "
              f"{r['acc_core']:>6.3f} {r['acc_atypical']:>6.3f} {r['drop_atypical']:>6.3f}")

    # Plot
    top15 = rdf.head(15)
    x = np.arange(len(top15))
    fig, ax = plt.subplots(figsize=(12,6))
    ax.bar(x-0.25, top15['acc_full'],     0.22, label='All symptoms',  color='#2E75B6')
    ax.bar(x,      top15['acc_core'],     0.22, label='Core only',     color='#27AE60')
    ax.bar(x+0.25, top15['acc_atypical'], 0.22, label='Atypical only', color='#C0392B')
    ax.set_xticks(x); ax.set_xticklabels(top15['disease'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Accuracy'); ax.set_title('TEST 5 — Atypical Presentation (top 15 fragile)')
    ax.legend(); ax.grid(alpha=0.3, axis='y'); ax.set_ylim(0,1.1)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/test5_atypical.png', dpi=200, bbox_inches='tight')
    plt.close()
    print(f"\n  Saved: test5_atypical.csv + .png\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 6 — RELATED SYMPTOM INJECTION
# Add extra symptoms that are plausible for the SAME disease
# (drawn from training frequency pool — these are medically valid extras).
# Compare against random injection to see if related > random hurts more.
# ══════════════════════════════════════════════════════════════════════════════
def test6_related_injection():
    print("="*65)
    print("  TEST 6: Related Symptom Injection")
    print("  (Add plausible extras from same disease pool vs random extras)")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)
    rng   = np.random.RandomState(42)
    rows  = []
    print(f"\n  {'Scenario':<48} {'Acc':>7} {'Top3':>7} {'F1':>7}")
    print(f"  {'-'*71}")

    for n_add in [1, 2, 3, 5]:
        # ── Related: symptoms from same disease's training pool ────────────
        X_rel = X_all.copy()
        for i in range(len(X_rel)):
            disease     = y_raw_all[i]
            pool        = DSYM_POOL.get(disease, [])
            absent_pool = [s for s in pool
                           if s in SYM_SET and X_rel[i, SYMPTOMS.index(s)]==0]
            if len(absent_pool) >= n_add:
                chosen = rng.choice(absent_pool, size=n_add, replace=False)
                for s in chosen:
                    X_rel[i, SYMPTOMS.index(s)] = 1.0

        probs = infer(X_rel)
        preds = np.argmax(probs,1)
        acc   = (preds==y_all).mean()
        top3  = top_k_accuracy_score(y_all, probs, k=3, labels=range(NC))
        f1    = f1_score(y_all, preds, average='macro', zero_division=0)
        label = f'Add {n_add} RELATED (same-disease pool) symptoms'
        print(f"  {label:<48} {acc:>7.4f} {top3:>7.4f} {f1:>7.4f}")
        rows.append({'scenario':label,'acc':acc,'top3':top3,'f1':f1})

        # ── Random: symptoms from ANY position not currently active ────────
        X_rnd = X_all.copy()
        for i in range(len(X_rnd)):
            off_idx = np.where(X_rnd[i]==0)[0]
            if len(off_idx)>=n_add:
                X_rnd[i, rng.choice(off_idx, size=n_add, replace=False)] = 1.0

        probs = infer(X_rnd)
        preds = np.argmax(probs,1)
        acc   = (preds==y_all).mean()
        top3  = top_k_accuracy_score(y_all, probs, k=3, labels=range(NC))
        f1    = f1_score(y_all, preds, average='macro', zero_division=0)
        label = f'Add {n_add} RANDOM  (any position)         symptoms'
        print(f"  {label:<48} {acc:>7.4f} {top3:>7.4f} {f1:>7.4f}")
        rows.append({'scenario':label,'acc':acc,'top3':top3,'f1':f1})

    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{RESULTS_DIR}/test6_related_injection.csv', index=False)
    print(f"\n  Saved: test6_related_injection.csv\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 7 — CROSS-CONTAMINATION
# Inject symptoms from the MOST SIMILAR (confusable) disease.
# Simulates: patient has Disease A but also shows hallmark symptoms of Disease B.
# ══════════════════════════════════════════════════════════════════════════════
def test7_cross_contamination():
    print("="*65)
    print("  TEST 7: Cross-Contamination")
    print("  (Inject hallmark symptoms of the most confusable similar disease)")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)
    rng   = np.random.RandomState(42)

    # Lookup: disease -> its most confusable partner + that partner's unique symptoms
    best_confuse = {}
    for d1, d2, jac, shared, only1, only2 in CONFUSABLE:
        if d1 not in best_confuse: best_confuse[d1] = (d2, only2)  # inject d2's unique syms
        if d2 not in best_confuse: best_confuse[d2] = (d1, only1)

    rows = []
    print(f"\n  {'Disease':<32} {'Confused with':<32} "
          f"{'Clean':>7} {'+2':>7} {'+5':>7}")
    print(f"  {'-'*85}")

    for disease_raw in sorted(np.unique(y_raw_all)):
        enc = auto_map(disease_raw)
        if not enc or enc not in le.classes_: continue
        if disease_raw not in best_confuse: continue
        idx  = np.where(le.classes_==enc)[0][0]
        mask = y_all==idx
        if mask.sum()==0: continue

        confuse_disease, inject_syms = best_confuse[disease_raw]
        # Only inject symptoms that exist in our SYMPTOMS list
        inject_pool = [s for s in inject_syms if s in SYM_SET]

        X_sub = X_all[mask]
        acc_clean = (np.argmax(infer(X_sub),1)==idx).mean()

        accs = []
        for n_add in [2, 5]:
            X_aug = X_sub.copy()
            pool_use = inject_pool[:] if inject_pool else []
            if len(pool_use) >= n_add:
                for i in range(len(X_aug)):
                    chosen = rng.choice(pool_use, size=n_add, replace=False)
                    for s in chosen:
                        X_aug[i, SYMPTOMS.index(s)] = 1.0
            acc = (np.argmax(infer(X_aug),1)==idx).mean()
            accs.append(acc)

        print(f"  {disease_raw:<32} {confuse_disease:<32} "
              f"{acc_clean:>7.3f} {accs[0]:>7.3f} {accs[1]:>7.3f}")
        rows.append({'disease':disease_raw,'confused_with':confuse_disease,
                     'n_samples':int(mask.sum()),'acc_clean':round(acc_clean,4),
                     'acc_plus2':round(accs[0],4),'acc_plus5':round(accs[1],4),
                     'drop_plus5':round(acc_clean-accs[1],4)})

    rdf = pd.DataFrame(rows).sort_values('drop_plus5', ascending=False)
    rdf.to_csv(f'{RESULTS_DIR}/test7_cross_contamination.csv', index=False)
    print(f"\n  Saved: test7_cross_contamination.csv\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 8 — EARLY / MID / LATE STAGE
# Drop symptoms for early stage; add related extras for late stage.
# ══════════════════════════════════════════════════════════════════════════════
def test8_disease_stage():
    print("="*65)
    print("  TEST 8: Disease Stage Simulation")
    print("  (Early=2 core symptoms | Mid=50% of symptoms | Late=all+3 related)")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)
    rng   = np.random.RandomState(42)

    X_early = np.zeros_like(X_all)
    X_mid   = np.zeros_like(X_all)
    X_late  = X_all.copy()

    for i in range(len(X_all)):
        disease  = y_raw_all[i]
        freq_map = DSYM_FREQ.get(disease, {})
        on_idx   = np.where(X_all[i]==1)[0]
        ranked   = sorted(on_idx, key=lambda j: freq_map.get(SYMPTOMS[j],0), reverse=True)

        # Early: top 2 most frequent symptoms only
        X_early[i, ranked[:2]] = 1.0

        # Mid: top half
        X_mid[i, ranked[:max(1,len(ranked)//2)]] = 1.0

        # Late: all + up to 3 related extras (from disease pool, not already present)
        pool = [s for s in DSYM_POOL.get(disease, [])
                if s in SYM_SET and X_late[i, SYMPTOMS.index(s)]==0]
        if len(pool)>=3:
            for s in rng.choice(pool, size=3, replace=False):
                X_late[i, SYMPTOMS.index(s)] = 1.0

    print()
    ms = [
        score(y_all, infer(X_all),   'Baseline (as-is)'),
        score(y_all, infer(X_early), 'Early stage  — top 2 symptoms only'),
        score(y_all, infer(X_mid),   'Mid stage    — top 50% of symptoms'),
        score(y_all, infer(X_late),  'Late stage   — all + 3 related extras'),
    ]
    pd.DataFrame(ms).to_csv(f'{RESULTS_DIR}/test8_stages.csv', index=False)

    labels = ['Baseline','Early','Mid','Late']
    bar_chart(labels,
              [m['acc'] for m in ms], [m['f1'] for m in ms],
              'TEST 8 — Disease Stage Simulation',
              f'{RESULTS_DIR}/test8_stages.png')
    print(f"\n  Saved: test8_stages.csv + .png\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 9 — DANGEROUS MISCLASSIFICATION AUDIT
# For critical diseases: when the model is wrong, what does it say instead?
# A Heart Attack predicted as GERD is catastrophic.
# ══════════════════════════════════════════════════════════════════════════════
def test9_dangerous_misclass():
    print("="*65)
    print("  TEST 9: Dangerous Misclassification Audit")
    print("  (For critical diseases — what does wrong prediction look like?)")
    print("="*65)

    CRITICAL = {
        'Heart attack'               : 'Life-threatening',
        'Paralysis (brain hemorrhage)': 'Life-threatening',
        'AIDS'                       : 'Life-threatening',
        'Pneumonia'                  : 'High-risk',
        'Tuberculosis'               : 'High-risk',
        'Dengue'                     : 'High-risk',
        'Malaria'                    : 'High-risk',
        'Hepatitis B'                : 'High-risk',
        'Hepatitis D'                : 'High-risk',
        'Hepatitis E'                : 'High-risk',
    }

    df = pd.read_csv(TEST_DATASET)
    X_all, y_raw_all, _ = wide_to_binary(df)
    y_all  = le.transform(y_raw_all)
    probs  = infer(X_all)
    preds  = np.argmax(probs, 1)

    all_rows = []
    print()
    for disease_raw, severity in CRITICAL.items():
        enc = auto_map(disease_raw)
        if not enc or enc not in le.classes_: continue
        idx  = np.where(le.classes_==enc)[0][0]
        mask = y_all==idx
        if mask.sum()==0:
            print(f"  {disease_raw}: not in test set"); continue

        correct   = (preds[mask]==idx).mean()
        wrong_preds = preds[mask][preds[mask]!=idx]

        print(f"\n  [{severity}] {disease_raw}  "
              f"(n={mask.sum()}, accuracy={correct:.4f})")

        if len(wrong_preds)==0:
            print(f"    -> Perfect — no misclassifications")
        else:
            for wrong_class, cnt in Counter(wrong_preds).most_common(5):
                wrong_name = le.classes_[wrong_class]
                # max confidence the model had when it made this wrong call
                wrong_mask = (preds[mask]!=idx)
                wrong_conf = probs[mask][wrong_mask][:,wrong_class].max()
                danger = "!!DANGER!!" if severity=='Life-threatening' else "!"
                print(f"    {danger} Predicted as: {wrong_name:<35} "
                      f"count={cnt}  max_conf={wrong_conf:.3f}")
                all_rows.append({'severity':severity,'true':disease_raw,
                                 'predicted_as':wrong_name,'count':cnt,
                                 'max_conf_when_wrong':round(wrong_conf,4),
                                 'true_acc':round(correct,4)})

    rdf = pd.DataFrame(all_rows)
    rdf.to_csv(f'{RESULTS_DIR}/test9_dangerous_misclass.csv', index=False)
    print(f"\n\n  Saved: test9_dangerous_misclass.csv\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 10 — SYMPTOM DROPOUT (per-symptom importance)
# Remove one symptom at a time — which symptoms are most diagnostic?
# ══════════════════════════════════════════════════════════════════════════════
def test10_symptom_dropout(n_samples=300):
    print("="*65)
    print("  TEST 10: Symptom Dropout — Per-Symptom Importance")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X_full, y_raw, _ = wide_to_binary(df)
    y = le.transform(y_raw)
    idx = np.random.RandomState(42).choice(len(y), size=min(n_samples,len(y)), replace=False)
    X, y = X_full[idx], y[idx]
    base = (np.argmax(infer(X),1)==y).mean()
    print(f"  Baseline ({len(y)} samples): {base:.4f}\n  Running 132 symptom drops...")
    rows = []
    for i, sym in enumerate(SYMPTOMS):
        Xd = X.copy(); Xd[:,i] = 0.0
        acc = (np.argmax(infer(Xd),1)==y).mean()
        rows.append({'symptom':sym,'acc_after_drop':round(acc,4),
                     'acc_drop':round(base-acc,4)})
    rdf = pd.DataFrame(rows).sort_values('acc_drop', ascending=False)
    rdf.to_csv(f'{RESULTS_DIR}/test10_dropout.csv', index=False)
    print(f"\n  Top 15 most critical symptoms:")
    print(f"  {'Symptom':<35} {'Acc Drop':>10}")
    print(f"  {'-'*47}")
    for _, r in rdf.head(15).iterrows():
        bar = '#' * max(0, int(r['acc_drop']*200))
        print(f"  {r['symptom']:<35} {r['acc_drop']:>10.4f}  {bar}")
    top20 = rdf.head(20)
    plt.figure(figsize=(10,6))
    plt.barh(top20['symptom'][::-1], top20['acc_drop'][::-1], color='#C0392B', alpha=0.85)
    plt.xlabel('Accuracy Drop when symptom removed')
    plt.title('TEST 10 — Top 20 Most Critical Symptoms')
    plt.grid(alpha=0.3, axis='x'); plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/test10_dropout.png', dpi=200, bbox_inches='tight')
    plt.close()
    print(f"\n  Saved: test10_dropout.csv + .png\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 11 — PROGRESSIVE UNRELATED NOISE
# Add increasing numbers of random symptoms that are NOT related to the disease.
# ══════════════════════════════════════════════════════════════════════════════
def test11_noise():
    print("="*65)
    print("  TEST 11: Progressive Unrelated Noise")
    print("  (Add random symptoms unrelated to the disease — simulates noisy input)")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X, y_raw, _ = wide_to_binary(df)
    y   = le.transform(y_raw)
    rng = np.random.RandomState(42)
    rows = []
    print(f"\n  {'Noise level':<15} {'Acc':>8} {'Top3':>8} {'F1':>8}")
    print(f"  {'-'*42}")
    for n in [0,1,2,3,5,8,10,15,20]:
        Xn = X.copy()
        if n>0:
            for i in range(len(Xn)):
                disease = y_raw[i]
                # Exclude symptoms from the disease's own pool — truly unrelated
                disease_pool = set(DSYM_POOL.get(disease, []))
                candidates   = [j for j in range(len(SYMPTOMS))
                                if Xn[i,j]==0 and SYMPTOMS[j] not in disease_pool]
                if len(candidates)>=n:
                    Xn[i, rng.choice(candidates, size=n, replace=False)] = 1.0
        probs = infer(Xn)
        preds = np.argmax(probs,1)
        acc   = (preds==y).mean()
        top3  = top_k_accuracy_score(y, probs, k=3, labels=range(NC))
        f1    = f1_score(y, preds, average='macro', zero_division=0)
        print(f"  +{n:<13} {acc:>8.4f} {top3:>8.4f} {f1:>8.4f}")
        rows.append({'noise':n,'acc':round(acc,4),'top3':round(top3,4),'f1':round(f1,4)})
    rdf = pd.DataFrame(rows)
    rdf.to_csv(f'{RESULTS_DIR}/test11_noise.csv', index=False)
    plt.figure(figsize=(8,5))
    plt.plot(rdf['noise'],rdf['acc'],'b-o',ms=7,lw=2,label='Accuracy')
    plt.plot(rdf['noise'],rdf['top3'],'g-s',ms=7,lw=2,label='Top-3')
    plt.plot(rdf['noise'],rdf['f1'],'r-^',ms=7,lw=2,label='Macro F1')
    plt.xlabel('Unrelated symptoms added'); plt.ylabel('Score')
    plt.title('TEST 11 — Unrelated Noise Robustness')
    plt.legend(); plt.grid(alpha=0.3); plt.ylim(0,1.05); plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/test11_noise.png', dpi=200)
    plt.close()
    print(f"\n  Saved: test11_noise.csv + .png\n")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 12 — CALIBRATION + SPEED
# ══════════════════════════════════════════════════════════════════════════════
def test12_calibration_speed():
    print("="*65)
    print("  TEST 12: Confidence Calibration + Speed")
    print("="*65)
    df = pd.read_csv(TEST_DATASET)
    X, y_raw, _ = wide_to_binary(df)
    y     = le.transform(y_raw)
    probs = infer(X)
    confs = np.max(probs,1)
    correct = (np.argmax(probs,1)==y).astype(float)

    # Calibration
    bins = np.arange(0.5,1.01,0.05)
    cal_rows = []
    print(f"\n  {'Bin':<12} {'N':>6} {'Accuracy':>10} {'Avg Conf':>10}")
    print(f"  {'-'*42}")
    for i in range(len(bins)-1):
        lo,hi = bins[i],bins[i+1]
        m     = (confs>=lo)&(confs<hi)
        if m.sum()>0:
            acc=correct[m].mean(); conf=confs[m].mean()
            cal_rows.append({'bin':f'{lo:.2f}-{hi:.2f}','n':int(m.sum()),
                             'acc':round(acc,4),'avg_conf':round(conf,4)})
            print(f"  {lo:.2f}-{hi:.2f}   {m.sum():>6}     {acc:>8.4f}     {conf:>8.4f}")

    rdf = pd.DataFrame(cal_rows)
    rdf.to_csv(f'{RESULTS_DIR}/test12_calibration.csv', index=False)
    plt.figure(figsize=(7,5))
    plt.plot([0.5,1.0],[0.5,1.0],'k--',alpha=0.5,label='Perfect')
    plt.scatter(rdf['avg_conf'],rdf['acc'],s=rdf['n']*1.5,color='#2E75B6',alpha=0.8)
    plt.xlabel('Avg Confidence'); plt.ylabel('Accuracy')
    plt.title('TEST 12 — Confidence Calibration')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/test12_calibration.png', dpi=200)
    plt.close()

    # Speed
    print(f"\n  Speed benchmark:")
    X_dummy = np.random.randint(0,2,(1000,len(SYMPTOMS))).astype(float)
    sp_rows = []
    for bs in [1,8,32,64,128,256,512,1000]:
        times=[]
        for _ in range(5):
            t0=time.time(); infer(X_dummy[:bs]); times.append(time.time()-t0)
        avg=np.mean(times)
        sp_rows.append({'batch':bs,'time_s':round(avg,4),
                        'samp_per_sec':round(bs/avg),'ms_per_samp':round(avg/bs*1000,2)})
        print(f"  Batch {bs:>5} | {avg:.3f}s | {bs/avg:>6.0f} samp/sec | {avg/bs*1000:.2f} ms/samp")
    pd.DataFrame(sp_rows).to_csv(f'{RESULTS_DIR}/test12_speed.csv', index=False)
    print(f"\n  Saved: test12_calibration.csv + .png + test12_speed.csv\n")

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY REPORT
# ══════════════════════════════════════════════════════════════════════════════
def print_summary():
    print("="*65)
    print("  STRESS TEST COMPLETE — FILES SAVED")
    print("="*65)
    for f in sorted(os.listdir(RESULTS_DIR)):
        size = os.path.getsize(f'{RESULTS_DIR}/{f}')
        print(f"  {f:<45} {size:>8} bytes")
    print(f"\n  All results at: {RESULTS_DIR}")

# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    print("\n" + "="*65)
    print("        SYMPTOM MODEL — CLINICAL STRESS TEST SUITE")
    print("="*65 + "\n")

    test1_training_baseline()
    test2_external_baseline()
    test3_confusable_pairs()
    test4_min_symptoms()
    test5_atypical()
    test6_related_injection()
    test7_cross_contamination()
    test8_disease_stage()
    test9_dangerous_misclass()
    test10_symptom_dropout()
    test11_noise()
    test12_calibration_speed()
    print_summary()

Loading model...


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model loaded | 132 symptoms | 41 diseases
Known classes: ['AIDS', 'Acne', 'Alcoholic Hepatitis', 'Allergy', 'Arthritis'] ...

Disease mapping: 28 matched, 13 excluded
  Excluded (not in label encoder): ['Fungal infection', 'Chronic cholestasis', 'Peptic ulcer diseae', 'Cervical spondylosis', 'Chicken pox', 'hepatitis A', 'Alcoholic hepatitis', 'Dimorphic hemmorhoids(piles)', 'Heart attack', 'Varicose veins', 'Osteoarthristis', '(vertigo) Paroymsal  Positional Vertigo', 'Urinary tract infection']

        SYMPTOM MODEL — CLINICAL STRESS TEST SUITE


  TEST 1: Training Distribution Baseline (symbipredict_2022)
  Samples=4961  Speed=111 samp/sec

  [Baseline — symbipredict]
    Acc=1.0000  Top3=1.0000  Top5=1.0000  F1=1.0000  AvgConf=0.9799
  Saved: test1_train_cm.png

  TEST 2: External Dataset Baseline (dataset.csv — unmodified)
  Usable samples=3360

  [External baseline]
    Acc=1.0000  Top3=1.0000  Top5=1.0000  F1=1.0000  AvgConf=0.9997
  Saved: test2_per_class.csv + test2_per_diseas

In [4]:
"""
SYMPTOM MODEL V2 — ENHANCED STRESS TEST (BEST vs FINAL comparison)
===================================================================
15 tests on BOTH models → head-to-head comparison → deployment recommendation
Downloads: best_symptom_model.keras, final_symptom_model.keras,
           symptom_scaler.pkl, symptom_label_encoder.pkl, symptom_temperature.pkl
"""
import numpy as np, pandas as pd, pickle, time, os
import matplotlib.pyplot as plt, seaborn as sns
from tensorflow.keras.models import load_model
from sklearn.metrics import (classification_report, confusion_matrix,
                             top_k_accuracy_score, f1_score)
from itertools import combinations
from collections import Counter

# ─── Paths (each model has its own pkl files) ────────────────────────────────
DL = '/Users/ashishkishore/Downloads'

# Model A: best_symptom_model (from fold-based training)
BEST_MODEL_PATH   = f'{DL}/best_symptom_model.keras'
BEST_SCALER_PATH  = f'{DL}/symptom_scaler.pkl'
BEST_ENCODER_PATH = f'{DL}/symptom_label_encoder.pkl'
BEST_TEMP_PATH    = f'{DL}/symptom_temperature.pkl'

# Model B: final_symptom_model (from single-run training)
FINAL_MODEL_PATH   = f'{DL}/final_symptom_model.keras'
FINAL_SCALER_PATH  = f'{DL}/symptom_scaler (1).pkl'
FINAL_ENCODER_PATH = f'{DL}/symptom_label_encoder (1).pkl'
FINAL_TEMP_PATH    = f'{DL}/symptom_temperature (1).pkl'

TRAIN_DATASET = f'{DL}/symbipredict_2022.csv'
TEST_DATASET  = f'{DL}/dataset.csv'
RESULTS_DIR   = f'{DL}/stress_test_results_v2'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─── Verify files ────────────────────────────────────────────────────────────
print("Checking required files...")
ALL_PATHS = {
    'Best Model':    BEST_MODEL_PATH,
    'Best Scaler':   BEST_SCALER_PATH,
    'Best Encoder':  BEST_ENCODER_PATH,
    'Final Model':   FINAL_MODEL_PATH,
    'Final Scaler':  FINAL_SCALER_PATH,
    'Final Encoder': FINAL_ENCODER_PATH,
    'TrainCSV':      TRAIN_DATASET,
    'TestCSV':       TEST_DATASET,
}
OPTIONAL = {'Best Temp': BEST_TEMP_PATH, 'Final Temp': FINAL_TEMP_PATH}
missing = []
for name, path in {**ALL_PATHS, **OPTIONAL}.items():
    found = os.path.exists(path)
    status = "✅" if found else ("⚠️ optional" if name in OPTIONAL else "❌ MISSING")
    print(f"  {status}  {name}: {path}")
    if not found and name not in OPTIONAL:
        missing.append(f"{name}: {path}")
if missing:
    print(f"\n  ERROR: {len(missing)} required file(s) not found!")
    print(f"  Update the paths at the top of this script.")
    raise FileNotFoundError(f"Missing: {missing}")

# ─── Load ────────────────────────────────────────────────────────────────────
print("\nLoading models...")
best_model  = load_model(BEST_MODEL_PATH)
final_model = load_model(FINAL_MODEL_PATH)

best_scaler  = pickle.load(open(BEST_SCALER_PATH, 'rb'))
best_le      = pickle.load(open(BEST_ENCODER_PATH, 'rb'))
final_scaler = pickle.load(open(FINAL_SCALER_PATH, 'rb'))
final_le     = pickle.load(open(FINAL_ENCODER_PATH, 'rb'))

try:    best_temp = pickle.load(open(BEST_TEMP_PATH, 'rb'))
except: best_temp = 1.0
try:    final_temp = pickle.load(open(FINAL_TEMP_PATH, 'rb'))
except: final_temp = 1.0
print(f"  Best  temp: {best_temp:.2f}")
print(f"  Final temp: {final_temp:.2f}")

# Assets lookup — maps model object → its scaler, encoder, temperature
MODEL_ASSETS = {
    id(best_model):  {'scaler': best_scaler,  'le': best_le,  'temp': best_temp,  'name': 'best'},
    id(final_model): {'scaler': final_scaler, 'le': final_le, 'temp': final_temp, 'name': 'final'},
}

# Use the best_le as the primary encoder (both should be identical — same dataset)
le = best_le

train_df = pd.read_csv(TRAIN_DATASET)
SYMPTOMS = train_df.drop('prognosis', axis=1).columns.tolist()
NC       = len(le.classes_)
SYM_SET  = set(SYMPTOMS)
N_SYMS   = len(SYMPTOMS)
print(f"  {N_SYMS} symptoms | {NC} diseases | 2 models loaded")

# ─── Case-insensitive disease mapping ─────────────────────────────────────────
test_df_raw = pd.read_csv(TEST_DATASET)
le_classes_lower = {c.strip().lower(): c for c in le.classes_}

def auto_map(disease_str):
    ds = disease_str.strip()
    if ds in le.classes_: return ds
    dl = ds.lower()
    if dl in le_classes_lower: return le_classes_lower[dl]
    MANUAL = {
        '(vertigo) paroymsal  positional vertigo': 'Vertigo',
        'dimorphic hemmorhoids(piles)': 'Dimorphic Hemmorhoids (piles)',
        'peptic ulcer diseae': 'Peptic Ulcer Disease',
        'osteoarthristis': 'Osteoarthritis',
        'heart attack': 'Heart Attack', 'chicken pox': 'Chickenpox',
    }
    if dl in MANUAL and MANUAL[dl] in le.classes_: return MANUAL[dl]
    dl_clean = ''.join(c for c in dl if c.isalnum())
    for cls in le.classes_:
        if ''.join(c for c in cls.strip().lower() if c.isalnum()) == dl_clean: return cls
    return None

DISEASE_MAP = {}; unmapped = []
for d in test_df_raw['Disease'].unique():
    mapped = auto_map(d)
    if mapped: DISEASE_MAP[d] = mapped
    else: unmapped.append(d)
print(f"\n  Disease mapping: {len(DISEASE_MAP)} matched, {len(unmapped)} excluded")
if unmapped: print(f"  Excluded: {unmapped}")

# ─── Knowledge base ──────────────────────────────────────────────────────────
DSYM_FREQ, DSYM_POOL, DSYM_CORE, DSYM_RARE = {}, {}, {}, {}
for disease, group in train_df.groupby('prognosis'):
    X = group.drop('prognosis', axis=1).values.astype(float)
    freq = X.mean(axis=0)
    DSYM_FREQ[disease] = {SYMPTOMS[i]: freq[i] for i in range(N_SYMS) if freq[i] > 0}
    DSYM_POOL[disease] = [SYMPTOMS[i] for i in range(N_SYMS) if freq[i] >= 0.10]
    DSYM_CORE[disease] = [SYMPTOMS[i] for i in range(N_SYMS) if freq[i] >= 0.80]
    DSYM_RARE[disease] = [SYMPTOMS[i] for i in range(N_SYMS) if 0 < freq[i] < 0.40]

def build_confusable_pairs(top_n=20):
    pairs = []
    for d1, d2 in combinations(DSYM_POOL.keys(), 2):
        s1, s2 = set(DSYM_POOL[d1]), set(DSYM_POOL[d2])
        if not s1 | s2: continue
        j = len(s1 & s2) / len(s1 | s2)
        pairs.append((d1, d2, j, sorted(s1 & s2), sorted(s1 - s2), sorted(s2 - s1)))
    pairs.sort(key=lambda x: -x[2])
    return pairs[:top_n]
CONFUSABLE = build_confusable_pairs()

# ─── Core helpers (model-agnostic) ────────────────────────────────────────────
def wide_to_binary(df):
    sym_cols = [c for c in df.columns if c.startswith('Symptom_')]
    X = np.zeros((len(df), N_SYMS), dtype=float)
    sym_lower_map = {s.strip().lower(): i for i, s in enumerate(SYMPTOMS)}
    for idx, row in df.iterrows():
        for col in sym_cols:
            val = row[col]
            if isinstance(val, str):
                sym = val.strip().lower()
                if sym in sym_lower_map:
                    X[idx, sym_lower_map[sym]] = 1.0
                elif sym.replace(' ', '_') in sym_lower_map:
                    X[idx, sym_lower_map[sym.replace(' ', '_')]] = 1.0
    y_raw = df['Disease'].map(DISEASE_MAP)
    known = y_raw.notna()
    return X[known.values], y_raw[known].values, known

def apply_temperature(probs, T):
    if T == 1.0: return probs
    scaled = np.exp(np.log(probs + 1e-12) / T)
    return scaled / scaled.sum(axis=1, keepdims=True)

def infer_with(mdl, X_raw, use_temp=True):
    assets = MODEL_ASSETS[id(mdl)]
    probs = mdl.predict(np.expand_dims(assets['scaler'].transform(X_raw), -1), verbose=0)
    return apply_temperature(probs, assets['temp']) if use_temp else probs

def score_model(mdl, y, X_raw, tag=''):
    probs = infer_with(mdl, X_raw)
    preds = np.argmax(probs, 1)
    acc = (preds == y).mean()
    top3 = top_k_accuracy_score(y, probs, k=3, labels=range(NC))
    top5 = top_k_accuracy_score(y, probs, k=5, labels=range(NC))
    f1 = f1_score(y, preds, average='macro', zero_division=0)
    conf = np.max(probs, 1).mean()
    if tag: print(f"\n  [{tag}]")
    print(f"    Acc={acc:.4f}  Top3={top3:.4f}  Top5={top5:.4f}  F1={f1:.4f}  AvgConf={conf:.4f}")
    return dict(tag=tag, acc=acc, top3=top3, top5=top5, f1=f1, conf=conf)

# ─── Global score collector for final comparison ──────────────────────────────
COMPARISON = {'best': {}, 'final': {}}

def run_both(test_name, X_raw, y, tag_suffix=''):
    """Run inference on both models and record scores."""
    for label, mdl in [('best', best_model), ('final', final_model)]:
        probs = infer_with(mdl, X_raw)
        preds = np.argmax(probs, 1)
        acc = (preds == y).mean()
        top3 = top_k_accuracy_score(y, probs, k=3, labels=range(NC))
        f1 = f1_score(y, preds, average='macro', zero_division=0)
        conf = np.max(probs, 1).mean()
        tag = f"{label.upper()}{' — ' + tag_suffix if tag_suffix else ''}"
        print(f"    [{tag}] Acc={acc:.4f}  Top3={top3:.4f}  F1={f1:.4f}  Conf={conf:.4f}")
        key = f"{test_name}_{tag_suffix}" if tag_suffix else test_name
        COMPARISON[label][key] = {'acc': acc, 'top3': top3, 'f1': f1, 'conf': conf}
    return None

# ══════════════════════════════════════════════════════════════════════════════
# TEST 1 — TRAINING DISTRIBUTION BASELINE
# ══════════════════════════════════════════════════════════════════════════════
def test1_training_baseline():
    print("\n" + "=" * 65 + "\n  TEST 1: Training Distribution Baseline\n" + "=" * 65)
    df = pd.read_csv(TRAIN_DATASET)
    X = df.drop('prognosis', axis=1).values.astype(float)
    y = le.transform(df['prognosis'].values)
    run_both('T1', X, y, 'Train baseline')

# ══════════════════════════════════════════════════════════════════════════════
# TEST 2 — EXTERNAL DATASET BASELINE
# ══════════════════════════════════════════════════════════════════════════════
def test2_external_baseline():
    print("=" * 65 + "\n  TEST 2: External Dataset Baseline (dataset.csv)\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET)
    X, y_raw, _ = wide_to_binary(df); y = le.transform(y_raw)
    print(f"  Usable samples={len(y)}")
    run_both('T2', X, y, 'External')
    # Per-class report for both
    for label, mdl in [('best', best_model), ('final', final_model)]:
        probs = infer_with(mdl, X); preds = np.argmax(probs, 1)
        present = np.unique(y); names = le.classes_[present]
        report = classification_report(y, preds, labels=present,
                                       target_names=names, output_dict=True, zero_division=0)
        pd.DataFrame(report).T.to_csv(f'{RESULTS_DIR}/test2_{label}_per_class.csv')

# ══════════════════════════════════════════════════════════════════════════════
# TEST 3 — CONFUSABLE DISEASE PAIRS
# ══════════════════════════════════════════════════════════════════════════════
def test3_confusable_pairs():
    print("=" * 65 + "\n  TEST 3: Confusable Disease Pairs\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET); X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)
    for label, mdl in [('best', best_model), ('final', final_model)]:
        rows = []
        print(f"\n  [{label.upper()}]")
        print(f"  {'Disease A':<30} {'Disease B':<30} {'Jac':>5} {'AccA':>6} {'AccB':>6}")
        for d1, d2, jac, shared, only1, only2 in CONFUSABLE:
            enc1 = auto_map(d1); enc2 = auto_map(d2)
            if not enc1 or not enc2: continue
            idx1 = np.where(le.classes_ == enc1)[0][0]; idx2 = np.where(le.classes_ == enc2)[0][0]
            m1 = y_all == idx1; m2 = y_all == idx2
            if m1.sum() == 0 or m2.sum() == 0: continue
            p1 = np.argmax(infer_with(mdl, X_all[m1]), 1)
            p2 = np.argmax(infer_with(mdl, X_all[m2]), 1)
            acc1 = (p1 == idx1).mean(); acc2 = (p2 == idx2).mean()
            print(f"  {d1:<30} {d2:<30} {jac:>5.3f} {acc1:>6.3f} {acc2:>6.3f}")
            rows.append({'disease_a': d1, 'disease_b': d2, 'jaccard': jac,
                         'acc_a': round(acc1, 4), 'acc_b': round(acc2, 4)})
        pd.DataFrame(rows).to_csv(f'{RESULTS_DIR}/test3_{label}_confusable.csv', index=False)

# ══════════════════════════════════════════════════════════════════════════════
# TEST 3a — SYMPTOM ORDER SHUFFLE
# ══════════════════════════════════════════════════════════════════════════════
def test3a_shuffle():
    print("=" * 65 + "\n  TEST 3a: Symptom Order Shuffle\n" + "=" * 65)
    df = pd.read_csv(TRAIN_DATASET)
    X = df.drop('prognosis', axis=1).values.astype(float); y = le.transform(df['prognosis'].values)
    rng = np.random.RandomState(42)
    for label, mdl in [('best', best_model), ('final', final_model)]:
        probs = infer_with(mdl, X); acc0 = (np.argmax(probs, 1) == y).mean()
        print(f"\n  [{label.upper()}] Original: Acc={acc0:.4f}")
        for pct in [25, 50, 100]:
            X_s = X.copy(); n_cols = max(1, int(N_SYMS * pct / 100))
            for i in range(len(X_s)):
                cols = rng.choice(N_SYMS, size=n_cols, replace=False)
                rng.shuffle(X_s[i, cols])
            acc = (np.argmax(infer_with(mdl, X_s), 1) == y).mean()
            print(f"    Shuffle {pct}%: {acc:.4f} (drop={acc0 - acc:.4f})")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 4 — MINIMUM SYMPTOM THRESHOLD
# ══════════════════════════════════════════════════════════════════════════════
def test4_min_symptoms():
    print("=" * 65 + "\n  TEST 4: Minimum Symptom Threshold\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET); X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)
    for n_keep in [1, 2, 3, 5, 10, 'all']:
        X_aug = np.zeros_like(X_all)
        for i in range(len(X_all)):
            on = np.where(X_all[i] == 1)[0]
            if len(on) == 0: continue
            fm = DSYM_FREQ.get(y_raw_all[i], {})
            ranked = sorted(on, key=lambda j: fm.get(SYMPTOMS[j], 0), reverse=True)
            keep = ranked if n_keep == 'all' else ranked[:min(n_keep, len(ranked))]
            X_aug[i, keep] = 1.0
        tag = f'Top-{n_keep}' if n_keep != 'all' else 'All'
        run_both('T4', X_aug, y_all, tag)

# ══════════════════════════════════════════════════════════════════════════════
# TEST 5 — ATYPICAL PRESENTATION
# ══════════════════════════════════════════════════════════════════════════════
def test5_atypical():
    print("=" * 65 + "\n  TEST 5: Atypical Presentation\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET); X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)
    X_core = np.zeros_like(X_all); X_rare = np.zeros_like(X_all)
    for i in range(len(X_all)):
        disease = y_raw_all[i]; on = np.where(X_all[i] == 1)[0]
        fm = DSYM_FREQ.get(disease, {})
        core = [j for j in on if fm.get(SYMPTOMS[j], 0) >= 0.80]
        rare = [j for j in on if fm.get(SYMPTOMS[j], 0) < 0.40]
        if core: X_core[i, core] = 1.0
        else: X_core[i, on[:min(2, len(on))]] = 1.0
        if rare: X_rare[i, rare] = 1.0
        else: X_rare[i, on[-min(2, len(on)):]] = 1.0
    run_both('T5', X_all, y_all, 'Full')
    run_both('T5', X_core, y_all, 'Core only')
    run_both('T5', X_rare, y_all, 'Atypical/rare')

# ══════════════════════════════════════════════════════════════════════════════
# TEST 6 — RELATED SYMPTOM INJECTION
# ══════════════════════════════════════════════════════════════════════════════
def test6_related_injection():
    print("=" * 65 + "\n  TEST 6: Related Symptom Injection\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET); X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all); rng = np.random.RandomState(42)
    for n_add in [2, 5]:
        for mode in ['related', 'random']:
            Xm = X_all.copy()
            for i in range(len(Xm)):
                if mode == 'related':
                    pool = DSYM_POOL.get(y_raw_all[i], [])
                    cands = [s for s in pool if s in SYM_SET and Xm[i, SYMPTOMS.index(s)] == 0]
                    if len(cands) >= n_add:
                        for s in rng.choice(cands, size=n_add, replace=False):
                            Xm[i, SYMPTOMS.index(s)] = 1.0
                else:
                    cands = [j for j in range(N_SYMS) if Xm[i, j] == 0]
                    if len(cands) >= n_add:
                        Xm[i, rng.choice(cands, size=n_add, replace=False)] = 1.0
            run_both('T6', Xm, y_all, f'+{n_add} {mode}')

# ══════════════════════════════════════════════════════════════════════════════
# TEST 7 — CROSS-CONTAMINATION
# ══════════════════════════════════════════════════════════════════════════════
def test7_cross_contamination():
    print("=" * 65 + "\n  TEST 7: Cross-Contamination\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET); X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all); rng = np.random.RandomState(42)
    best_confuse = {}
    for d1, d2, jac, shared, only1, only2 in CONFUSABLE:
        if d1 not in best_confuse: best_confuse[d1] = (d2, only2)
        if d2 not in best_confuse: best_confuse[d2] = (d1, only1)
    for na in [2, 5]:
        Xa = X_all.copy()
        for i in range(len(Xa)):
            d = y_raw_all[i]
            if d not in best_confuse: continue
            _, pool = best_confuse[d]
            pool_valid = [s for s in pool if s in SYM_SET]
            if len(pool_valid) >= na:
                for s in rng.choice(pool_valid, size=na, replace=False):
                    Xa[i, SYMPTOMS.index(s)] = 1.0
        run_both('T7', Xa, y_all, f'+{na} confusable')

# ══════════════════════════════════════════════════════════════════════════════
# TEST 8 — DISEASE STAGE SIMULATION
# ══════════════════════════════════════════════════════════════════════════════
def test8_disease_stage():
    print("=" * 65 + "\n  TEST 8: Disease Stage Simulation\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET); X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all); rng = np.random.RandomState(42)
    X_early = np.zeros_like(X_all); X_mid = np.zeros_like(X_all); X_late = X_all.copy()
    for i in range(len(X_all)):
        d = y_raw_all[i]; fm = DSYM_FREQ.get(d, {}); on = np.where(X_all[i] == 1)[0]
        ranked = sorted(on, key=lambda j: fm.get(SYMPTOMS[j], 0), reverse=True)
        X_early[i, ranked[:2]] = 1.0
        X_mid[i, ranked[:max(1, len(ranked) // 2)]] = 1.0
        pool = [s for s in DSYM_POOL.get(d, []) if s in SYM_SET and X_late[i, SYMPTOMS.index(s)] == 0]
        if len(pool) >= 3:
            for s in rng.choice(pool, size=3, replace=False):
                X_late[i, SYMPTOMS.index(s)] = 1.0
    run_both('T8', X_all, y_all, 'Baseline')
    run_both('T8', X_early, y_all, 'Early (2 syms)')
    run_both('T8', X_mid, y_all, 'Mid (50%)')
    run_both('T8', X_late, y_all, 'Late (all+3)')

# ══════════════════════════════════════════════════════════════════════════════
# TEST 9 — DANGEROUS MISCLASSIFICATION AUDIT
# ══════════════════════════════════════════════════════════════════════════════
def test9_dangerous_misclass():
    print("=" * 65 + "\n  TEST 9: Dangerous Misclassification Audit\n" + "=" * 65)
    CRITICAL = {'Heart attack': 'Life-threatening', 'Paralysis (brain hemorrhage)': 'Life-threatening',
                'AIDS': 'Life-threatening', 'Pneumonia': 'High-risk', 'Tuberculosis': 'High-risk',
                'Dengue': 'High-risk', 'Malaria': 'High-risk', 'Hepatitis B': 'High-risk'}
    df = pd.read_csv(TEST_DATASET); X_all, y_raw_all, _ = wide_to_binary(df)
    y_all = le.transform(y_raw_all)
    for label, mdl in [('best', best_model), ('final', final_model)]:
        probs = infer_with(mdl, X_all); preds = np.argmax(probs, 1)
        rows = []
        print(f"\n  [{label.upper()}]")
        for d, sev in CRITICAL.items():
            enc = auto_map(d)
            if not enc or enc not in le.classes_: continue
            idx = np.where(le.classes_ == enc)[0][0]; mask = y_all == idx
            if mask.sum() == 0: continue
            correct = (preds[mask] == idx).mean()
            wrong = preds[mask][preds[mask] != idx]
            print(f"    [{sev}] {d}: acc={correct:.4f} ({mask.sum()} samples)")
            if len(wrong) > 0:
                for wc, cnt in Counter(wrong).most_common(3):
                    print(f"      → {le.classes_[wc]:<30} count={cnt}")
                    rows.append({'true': d, 'predicted': le.classes_[wc], 'count': cnt})
        pd.DataFrame(rows).to_csv(f'{RESULTS_DIR}/test9_{label}_misclass.csv', index=False)

# ══════════════════════════════════════════════════════════════════════════════
# TEST 10 — SYMPTOM DROPOUT
# ══════════════════════════════════════════════════════════════════════════════
def test10_symptom_dropout(n_samples=300):
    print("=" * 65 + "\n  TEST 10: Symptom Dropout\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET); X_full, y_raw, _ = wide_to_binary(df)
    y = le.transform(y_raw)
    idx = np.random.RandomState(42).choice(len(y), size=min(n_samples, len(y)), replace=False)
    X, y = X_full[idx], y[idx]
    for label, mdl in [('best', best_model), ('final', final_model)]:
        base = (np.argmax(infer_with(mdl, X), 1) == y).mean()
        rows = []
        for i, sym in enumerate(SYMPTOMS):
            Xd = X.copy(); Xd[:, i] = 0.0
            acc = (np.argmax(infer_with(mdl, Xd), 1) == y).mean()
            rows.append({'symptom': sym, 'acc_drop': round(base - acc, 4)})
        rdf = pd.DataFrame(rows).sort_values('acc_drop', ascending=False)
        rdf.to_csv(f'{RESULTS_DIR}/test10_{label}_dropout.csv', index=False)
        print(f"\n  [{label.upper()}] Baseline={base:.4f}  Top 5 critical symptoms:")
        for _, r in rdf.head(5).iterrows():
            print(f"    {r['symptom']:<35} drop={r['acc_drop']:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 11 — PROGRESSIVE NOISE
# ══════════════════════════════════════════════════════════════════════════════
def test11_noise():
    print("=" * 65 + "\n  TEST 11: Progressive Unrelated Noise\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET); X, y_raw, _ = wide_to_binary(df)
    y = le.transform(y_raw); rng = np.random.RandomState(42)
    for n in [0, 2, 5, 10, 20, 30]:
        Xn = X.copy()
        if n > 0:
            for i in range(len(Xn)):
                dp = set(DSYM_POOL.get(y_raw[i], []))
                cands = [j for j in range(N_SYMS) if Xn[i, j] == 0 and SYMPTOMS[j] not in dp]
                if len(cands) >= n: Xn[i, rng.choice(cands, size=n, replace=False)] = 1.0
        run_both('T11', Xn, y, f'+{n} noise')

# ══════════════════════════════════════════════════════════════════════════════
# TEST 12 — CALIBRATION + SPEED
# ══════════════════════════════════════════════════════════════════════════════
def test12_calibration_speed():
    print("=" * 65 + "\n  TEST 12: Calibration + Speed\n" + "=" * 65)
    df = pd.read_csv(TEST_DATASET); X, y_raw, _ = wide_to_binary(df)
    y = le.transform(y_raw)
    for label, mdl in [('best', best_model), ('final', final_model)]:
        probs = infer_with(mdl, X); confs = np.max(probs, 1)
        correct = (np.argmax(probs, 1) == y).astype(float)
        bins = np.arange(0.5, 1.01, 0.1); cal = []
        print(f"\n  [{label.upper()}] Calibration:")
        for i in range(len(bins) - 1):
            lo, hi = bins[i], bins[i + 1]; m = (confs >= lo) & (confs < hi)
            if m.sum() > 0:
                a = correct[m].mean(); c = confs[m].mean()
                print(f"    {lo:.1f}-{hi:.1f}: n={m.sum():>5}  acc={a:.4f}  conf={c:.4f}  gap={abs(c - a):.4f}")
                cal.append({'bin': f'{lo:.2f}-{hi:.2f}', 'n': int(m.sum()), 'acc': round(a, 4), 'conf': round(c, 4)})
        pd.DataFrame(cal).to_csv(f'{RESULTS_DIR}/test12_{label}_calibration.csv', index=False)
    # Speed for final model only
    print(f"\n  Speed:")
    Xd = np.random.randint(0, 2, (500, N_SYMS)).astype(float)
    for bs in [1, 32, 128, 500]:
        times = []
        for _ in range(3):
            t0 = time.time(); infer_with(final_model, Xd[:bs]); times.append(time.time() - t0)
        avg = np.mean(times)
        print(f"    Batch {bs:>4} | {avg:.3f}s | {bs / avg:>5.0f} samp/s | {avg / bs * 1000:.1f} ms/samp")

# ══════════════════════════════════════════════════════════════════════════════
# TEST 13 — COMORBIDITY
# ══════════════════════════════════════════════════════════════════════════════
def test13_comorbidity():
    print("=" * 65 + "\n  TEST 13: Comorbidity (2 diseases at once)\n" + "=" * 65)
    df = pd.read_csv(TRAIN_DATASET); rng = np.random.RandomState(42); n_trials = 100
    test_pairs = [('Diabetes ', 'Hypertension '), ('Malaria', 'Typhoid'),
                  ('Pneumonia', 'Bronchial Asthma'), ('Dengue', 'Malaria'),
                  ('Hepatitis B', 'Hepatitis C'), ('GERD', 'Peptic Ulcer Disease')]
    for label, mdl in [('best', best_model), ('final', final_model)]:
        print(f"\n  [{label.upper()}]")
        print(f"  {'A':<24} {'B':<24} {'A top1':>6} {'A top3':>6} {'B top1':>6} {'B top3':>6}")
        rows = []
        for dA, dB in test_pairs:
            if dA not in le.classes_ or dB not in le.classes_: continue
            idxA = np.where(le.classes_ == dA)[0][0]; idxB = np.where(le.classes_ == dB)[0][0]
            dfA = df[df['prognosis'] == dA].drop('prognosis', axis=1).values
            dfB = df[df['prognosis'] == dB].drop('prognosis', axis=1).values
            hA1, hA3, hB1, hB3 = 0, 0, 0, 0
            for _ in range(n_trials):
                combined = np.maximum(dfA[rng.randint(len(dfA))], dfB[rng.randint(len(dfB))])
                probs = infer_with(mdl, combined.reshape(1, -1))[0]
                t3 = np.argsort(probs)[-3:]
                if np.argmax(probs) == idxA: hA1 += 1
                if np.argmax(probs) == idxB: hB1 += 1
                if idxA in t3: hA3 += 1
                if idxB in t3: hB3 += 1
            print(f"  {dA:<24} {dB:<24} {hA1 / n_trials:>6.3f} {hA3 / n_trials:>6.3f} "
                  f"{hB1 / n_trials:>6.3f} {hB3 / n_trials:>6.3f}")
            rows.append({'a': dA, 'b': dB, 'a_t1': hA1 / n_trials, 'a_t3': hA3 / n_trials,
                         'b_t1': hB1 / n_trials, 'b_t3': hB3 / n_trials})
        pd.DataFrame(rows).to_csv(f'{RESULTS_DIR}/test13_{label}_comorbidity.csv', index=False)

# ══════════════════════════════════════════════════════════════════════════════
# TEST 14 — GRADUAL SYMPTOM ADDITION
# ══════════════════════════════════════════════════════════════════════════════
def test14_gradual_addition():
    print("=" * 65 + "\n  TEST 14: Gradual Symptom Addition\n" + "=" * 65)
    df = pd.read_csv(TRAIN_DATASET); rng = np.random.RandomState(42)
    test_diseases = ['Malaria', 'Pneumonia', 'Dengue', 'Heart Attack', 'AIDS', 'Diabetes ', 'Hepatitis B']
    for label, mdl in [('best', best_model), ('final', final_model)]:
        print(f"\n  [{label.upper()}]  {'Disease':<24} {'Lock-in N':>10} {'Total':>8}")
        rows = []
        for disease in test_diseases:
            if disease not in le.classes_: continue
            idx = np.where(le.classes_ == disease)[0][0]
            sub = df[df['prognosis'] == disease].drop('prognosis', axis=1).values
            sample = sub[rng.randint(len(sub))]
            on = np.where(sample == 1)[0]; fm = DSYM_FREQ.get(disease, {})
            ranked = sorted(on, key=lambda j: fm.get(SYMPTOMS[j], 0), reverse=True)
            lock_in = -1
            for n in range(1, len(ranked) + 1):
                x = np.zeros(N_SYMS); x[ranked[:n]] = 1.0
                pred = np.argmax(infer_with(mdl, x.reshape(1, -1)), 1)[0]
                if pred == idx and lock_in < 0: lock_in = n
                elif pred != idx: lock_in = -1
            if lock_in < 0: lock_in = len(ranked)
            print(f"           {disease:<24} {lock_in:>10} {len(ranked):>8}")
            rows.append({'disease': disease, 'lock_in': lock_in, 'total': len(ranked)})
        pd.DataFrame(rows).to_csv(f'{RESULTS_DIR}/test14_{label}_gradual.csv', index=False)

# ══════════════════════════════════════════════════════════════════════════════
# TEST 15 — WORST-CASE PER DISEASE
# ══════════════════════════════════════════════════════════════════════════════
def test15_worst_case():
    print("=" * 65 + "\n  TEST 15: Worst-Case Per Disease\n" + "=" * 65)
    df = pd.read_csv(TRAIN_DATASET); rng = np.random.RandomState(42)
    for label, mdl in [('best', best_model), ('final', final_model)]:
        print(f"\n  [{label.upper()}]  {'Disease':<28} {'Full':>6} {'Drop50':>7} {'Atyp':>6} {'+5N':>6} {'Worst':>6}")
        rows = []
        for disease in sorted(le.classes_):
            idx = np.where(le.classes_ == disease)[0][0]
            sub = df[df['prognosis'] == disease].drop('prognosis', axis=1).values
            if len(sub) == 0: continue
            fm = DSYM_FREQ.get(disease, {})
            acc_full = (np.argmax(infer_with(mdl, sub), 1) == idx).mean()
            Xd = sub.copy()
            for i in range(len(Xd)):
                on = np.where(Xd[i] == 1)[0]
                if len(on) > 2:
                    nd = max(1, len(on) // 2); Xd[i, rng.choice(on, size=nd, replace=False)] = 0
            acc_drop = (np.argmax(infer_with(mdl, Xd), 1) == idx).mean()
            Xa = sub.copy(); core = set(DSYM_CORE.get(disease, []))
            for i in range(len(Xa)):
                on = np.where(Xa[i] == 1)[0]; nc = [j for j in on if SYMPTOMS[j] not in core]
                if len(nc) >= 2:
                    for j in on:
                        if SYMPTOMS[j] in core: Xa[i, j] = 0
            acc_atyp = (np.argmax(infer_with(mdl, Xa), 1) == idx).mean()
            Xn = sub.copy(); dp = set(DSYM_POOL.get(disease, []))
            for i in range(len(Xn)):
                cands = [j for j in range(N_SYMS) if Xn[i, j] == 0 and SYMPTOMS[j] not in dp]
                if len(cands) >= 5: Xn[i, rng.choice(cands, size=5, replace=False)] = 1
            acc_noise = (np.argmax(infer_with(mdl, Xn), 1) == idx).mean()
            worst = min(acc_full, acc_drop, acc_atyp, acc_noise)
            print(f"           {disease:<28} {acc_full:>6.3f} {acc_drop:>7.3f} {acc_atyp:>6.3f} {acc_noise:>6.3f} {worst:>6.3f}")
            rows.append({'disease': disease, 'full': round(acc_full, 4), 'drop50': round(acc_drop, 4),
                         'atypical': round(acc_atyp, 4), 'noise5': round(acc_noise, 4), 'worst': round(worst, 4)})
        rdf = pd.DataFrame(rows).sort_values('worst')
        rdf.to_csv(f'{RESULTS_DIR}/test15_{label}_worst.csv', index=False)
        print(f"           Worst < 0.50: {(rdf['worst'] < 0.50).sum()}/{len(rdf)}")
        print(f"           Worst < 0.70: {(rdf['worst'] < 0.70).sum()}/{len(rdf)}")

# ══════════════════════════════════════════════════════════════════════════════
# FINAL COMPARISON & DEPLOYMENT RECOMMENDATION
# ══════════════════════════════════════════════════════════════════════════════
def final_comparison():
    print("\n" + "=" * 70)
    print("  🏆 BEST vs FINAL — HEAD-TO-HEAD COMPARISON")
    print("=" * 70)

    # Collect all matching test keys
    all_keys = sorted(set(COMPARISON['best'].keys()) & set(COMPARISON['final'].keys()))
    if not all_keys:
        print("  No comparison data collected!"); return

    # Build comparison table
    rows = []
    best_wins, final_wins, ties = 0, 0, 0
    for key in all_keys:
        b = COMPARISON['best'][key]
        f = COMPARISON['final'][key]
        diff_acc = f['acc'] - b['acc']
        diff_f1 = f['f1'] - b['f1']
        if abs(diff_acc) < 0.005: ties += 1; winner = 'TIE'
        elif diff_acc > 0: final_wins += 1; winner = 'FINAL'
        else: best_wins += 1; winner = 'BEST'
        rows.append({'test': key, 'best_acc': b['acc'], 'final_acc': f['acc'],
                     'best_f1': b['f1'], 'final_f1': f['f1'],
                     'acc_diff': round(diff_acc, 4), 'f1_diff': round(diff_f1, 4),
                     'winner': winner})

    cdf = pd.DataFrame(rows)
    cdf.to_csv(f'{RESULTS_DIR}/model_comparison.csv', index=False)

    # Print summary table
    print(f"\n  {'Test Scenario':<35} {'Best Acc':>8} {'Final Acc':>9} {'Diff':>7} {'Winner':>8}")
    print(f"  {'-' * 70}")
    for _, r in cdf.iterrows():
        arrow = '→' if r['winner'] == 'TIE' else ('✅' if r['winner'] == 'FINAL' else '⬅️')
        print(f"  {r['test']:<35} {r['best_acc']:>8.4f} {r['final_acc']:>9.4f} "
              f"{r['acc_diff']:>+7.4f} {arrow} {r['winner']}")

    # Aggregate scores
    avg_best_acc = np.mean([COMPARISON['best'][k]['acc'] for k in all_keys])
    avg_final_acc = np.mean([COMPARISON['final'][k]['acc'] for k in all_keys])
    avg_best_f1 = np.mean([COMPARISON['best'][k]['f1'] for k in all_keys])
    avg_final_f1 = np.mean([COMPARISON['final'][k]['f1'] for k in all_keys])

    print(f"\n  {'AGGREGATE':=^70}")
    print(f"  {'Metric':<30} {'Best Model':>15} {'Final Model':>15}")
    print(f"  {'-' * 62}")
    print(f"  {'Average Accuracy':<30} {avg_best_acc:>15.4f} {avg_final_acc:>15.4f}")
    print(f"  {'Average F1':<30} {avg_best_f1:>15.4f} {avg_final_f1:>15.4f}")
    print(f"  {'Tests Won':<30} {best_wins:>15} {final_wins:>15}")
    print(f"  {'Ties (<0.5% diff)':<30} {'':>15} {ties:>15}")

    # Deployment recommendation
    print(f"\n  {'DEPLOYMENT RECOMMENDATION':=^70}")
    # Score: weighted by robustness tests being more important
    # Final model is trained on MORE data (all train+val), so generally preferred if close
    score_best = avg_best_acc * 0.4 + avg_best_f1 * 0.4 + (best_wins / max(1, len(all_keys))) * 0.2
    score_final = avg_final_acc * 0.4 + avg_final_f1 * 0.4 + (final_wins / max(1, len(all_keys))) * 0.2

    if score_final >= score_best - 0.01:  # tie-break in favor of final (more training data)
        recommended = 'FINAL'
        reason = ("The FINAL model was trained on ALL data (train+val combined), "
                  "giving it maximum exposure to disease patterns. "
                  "It is the recommended choice for production deployment.")
    else:
        recommended = 'BEST'
        reason = ("The BEST model scored higher across robustness tests, "
                  "suggesting better generalization despite seeing less training data. "
                  "Consider using this for deployment.")

    print(f"\n  Composite Score — Best: {score_best:.4f}  |  Final: {score_final:.4f}")
    print(f"\n  🚀 RECOMMENDED FOR DEPLOYMENT: ** {recommended} MODEL **")
    print(f"\n  Reason: {reason}")
    if recommended == 'FINAL':
        print(f"\n  Deploy: final_symptom_model.keras")
    else:
        print(f"\n  Deploy: best_symptom_model.keras")
    print(f"  With:   symptom_scaler.pkl + symptom_label_encoder.pkl + symptom_temperature.pkl")

    # Save recommendation
    with open(f'{RESULTS_DIR}/deployment_recommendation.txt', 'w') as f:
        f.write(f"RECOMMENDED MODEL: {recommended}\n")
        f.write(f"Composite — Best: {score_best:.4f} | Final: {score_final:.4f}\n")
        f.write(f"Avg Acc   — Best: {avg_best_acc:.4f} | Final: {avg_final_acc:.4f}\n")
        f.write(f"Avg F1    — Best: {avg_best_f1:.4f} | Final: {avg_final_f1:.4f}\n")
        f.write(f"Wins      — Best: {best_wins} | Final: {final_wins} | Ties: {ties}\n")
        f.write(f"\n{reason}\n")
    print(f"\n  Saved: model_comparison.csv + deployment_recommendation.txt")

# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
def print_summary():
    print("\n" + "=" * 65 + "\n  ALL FILES SAVED\n" + "=" * 65)
    for f in sorted(os.listdir(RESULTS_DIR)):
        print(f"  {f:<50} {os.path.getsize(f'{RESULTS_DIR}/{f}'):>8} bytes")
    print(f"\n  Results: {RESULTS_DIR}")

# ══════════════════════════════════════════════════════════════════════════════
# RUN ALL
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    print("\n" + "=" * 70)
    print("  SYMPTOM MODEL V2 — BEST vs FINAL STRESS TEST (15 tests × 2 models)")
    print("=" * 70 + "\n")

    test1_training_baseline()
    test2_external_baseline()
    test3_confusable_pairs()
    test3a_shuffle()
    test4_min_symptoms()
    test5_atypical()
    test6_related_injection()
    test7_cross_contamination()
    test8_disease_stage()
    test9_dangerous_misclass()
    test10_symptom_dropout()
    test11_noise()
    test12_calibration_speed()
    test13_comorbidity()
    test14_gradual_addition()
    test15_worst_case()

    # HEAD-TO-HEAD COMPARISON
    final_comparison()
    print_summary()

Checking required files...
  ✅  Best Model: /Users/ashishkishore/Downloads/best_symptom_model.keras
  ✅  Best Scaler: /Users/ashishkishore/Downloads/symptom_scaler.pkl
  ✅  Best Encoder: /Users/ashishkishore/Downloads/symptom_label_encoder.pkl
  ✅  Final Model: /Users/ashishkishore/Downloads/final_symptom_model.keras
  ✅  Final Scaler: /Users/ashishkishore/Downloads/symptom_scaler (1).pkl
  ✅  Final Encoder: /Users/ashishkishore/Downloads/symptom_label_encoder (1).pkl
  ✅  TrainCSV: /Users/ashishkishore/Downloads/symbipredict_2022.csv
  ✅  TestCSV: /Users/ashishkishore/Downloads/dataset.csv
  ✅  Best Temp: /Users/ashishkishore/Downloads/symptom_temperature.pkl
  ✅  Final Temp: /Users/ashishkishore/Downloads/symptom_temperature (1).pkl

Loading models...


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  Best  temp: 0.50
  Final temp: 0.50
  132 symptoms | 41 diseases | 2 models loaded

  Disease mapping: 41 matched, 0 excluded

  SYMPTOM MODEL V2 — BEST vs FINAL STRESS TEST (15 tests × 2 models)


  TEST 1: Training Distribution Baseline
    [BEST — Train baseline] Acc=1.0000  Top3=1.0000  F1=1.0000  Conf=0.9907
    [FINAL — Train baseline] Acc=1.0000  Top3=1.0000  F1=1.0000  Conf=0.9998
  TEST 2: External Dataset Baseline (dataset.csv)
  Usable samples=4920
    [BEST — External] Acc=1.0000  Top3=1.0000  F1=1.0000  Conf=0.9907
    [FINAL — External] Acc=1.0000  Top3=1.0000  F1=1.0000  Conf=0.9998
  TEST 3: Confusable Disease Pairs

  [BEST]
  Disease A                      Disease B                        Jac   AccA   AccB
  Hepatitis D                    Hepatitis E                    0.692  1.000  1.000
  Hepatitis A                    Hepatitis D                    0.667  1.000  1.000
  Chronic Cholestasis            Hepatitis D                    0.600  1.000  1.000
  Chronic Ch